# 08_session_access_rule — base an access decision on the caller's session id

AOA keeps no session store and has no logout. It carries the session id; the application decides what a session may do. Here the application keeps a registry of ended sessions: `LogoutAction` records the caller's session, and the `@access_decide` rule of `ListOrdersAction` refuses a session found there.

Script version: `examples/step_13_fastapi/08_session_access_rule.py`. Guide: `docs/extensions/jwt.md`, section "Use the session id in an access rule".

In [ ]:
!pip install "aoa-action-machine[jwt]" aoa-fastapi-adapter

In [1]:
from datetime import UTC, datetime, timedelta
from itertools import count
import jwt
from fastapi.testclient import TestClient
from pydantic import Field
from aoa.action_machine.auth import ApplicationRole, GuestRole, NoAuthCoordinator
from aoa.action_machine.auth.jwt_auth import JwtAuthCoordinator
from aoa.action_machine.context import Context, ContextView, Ctx
from aoa.action_machine.domain.base_domain import BaseDomain
from aoa.action_machine.intents.access_control import Allowed, Refused, Verdict
from aoa.action_machine.intents.access_decide import access_decide
from aoa.action_machine.intents.aspects import summary_aspect
from aoa.action_machine.intents.check_roles import check_roles
from aoa.action_machine.intents.connection import connection
from aoa.action_machine.intents.context_requires import context_requires
from aoa.action_machine.intents.meta import meta
from aoa.action_machine.model import BaseAction, BaseParams, BaseResult, ParamsStub
from aoa.action_machine.model.base_state import BaseState
from aoa.action_machine.resources import Connections
from aoa.action_machine.resources.base_resource import BaseResource
from aoa.action_machine.runtime.action_product_machine import ActionProductMachine
from aoa.action_machine.runtime.tools_box import ToolsBox
from aoa.fastapi import FastApiAdapter

## The domain, the role and the session registry

`SessionRegistryResource` is ordinary application state. AOA does not know what it means; the Actions below use it through a declared connection.

In [2]:
_SECRET_KEY = "demo-only-secret-do-not-use-in-production"

_ALGORITHM = "HS256"

_TOKEN_TTL = timedelta(minutes=30)

_session_numbers = count(1)


class OrdersDomain(BaseDomain):
    name = "orders"
    description = "Orders and the sessions that may read them"

class ViewerRole(ApplicationRole):
    name = "viewer"
    description = "Read-only access."

@meta(description="Application registry of ended login sessions.", domain=OrdersDomain)
class SessionRegistryResource(BaseResource):
    def __init__(self) -> None:
        self._ended: set[str] = set()

    def get_wrapper_class(self) -> type[BaseResource] | None:
        return None  # in-memory registry, no transaction to wrap

    async def end(self, session_id: str) -> None:
        self._ended.add(session_id)

    async def is_ended(self, session_id: str) -> bool:
        return session_id in self._ended

## The Actions

`LogoutAction` reads the session id in an aspect and ends that session. `ListOrdersAction` declares the same field in its `@access_decide` method and refuses an ended session with the reason `SESSION_ENDED`.

In [3]:
class LoginParams(BaseParams):
    username: str = Field(description="Account username")

class TokenResult(BaseResult):
    access_token: str = Field(description="Signed JWT bearer token")

class LogoutResult(BaseResult):
    ended_session: str = Field(description="Session that was ended")

class OrdersResult(BaseResult):
    orders: list[str] = Field(description="Order identifiers visible to the caller")

In [4]:
@meta(description="Start a new login session and issue its token.", domain=OrdersDomain)
@check_roles(GuestRole)
class LoginAction(BaseAction[LoginParams, TokenResult]):

    @summary_aspect("Sign a token with a new session id.")
    async def login_summary(self, params: LoginParams, state: BaseState, box: ToolsBox, connections: Connections) -> TokenResult:
        now = datetime.now(UTC)
        payload = {
            "sub": params.username,
            "sid": f"session-{next(_session_numbers)}",
            "roles": ["viewer"],
            "iat": now,
            "exp": now + _TOKEN_TTL,
        }
        return TokenResult(access_token=jwt.encode(payload, _SECRET_KEY, algorithm=_ALGORITHM))

@meta(description="End the caller's current login session.", domain=OrdersDomain)
@check_roles(ViewerRole)
@connection(SessionRegistryResource, key="sessions")
class LogoutAction(BaseAction[ParamsStub, LogoutResult]):

    @summary_aspect("Record the current session as ended.")
    @context_requires(Ctx.User.session_id)
    async def logout_summary(self, params: ParamsStub, state: BaseState, box: ToolsBox, connections: Connections, ctx: ContextView) -> LogoutResult:
        session_id = ctx.get(Ctx.User.session_id)
        await connections["sessions"].end(session_id)
        return LogoutResult(ended_session=session_id)

@meta(description="List orders for a caller whose session is still active.", domain=OrdersDomain)
@check_roles(ViewerRole)
@connection(SessionRegistryResource, key="sessions")
class ListOrdersAction(BaseAction[ParamsStub, OrdersResult]):

    @access_decide("Refuse a session the application has ended")
    @context_requires(Ctx.User.session_id)
    async def active_session_access_decide(self, params: ParamsStub, box: ToolsBox, connections: Connections, ctx: ContextView) -> Verdict:
        if await connections["sessions"].is_ended(ctx.get(Ctx.User.session_id)):
            return Refused("SESSION_ENDED")
        return Allowed()

    @summary_aspect("Return the caller's orders.")
    async def list_orders_summary(self, params: ParamsStub, state: BaseState, box: ToolsBox, connections: Connections) -> OrdersResult:
        return OrdersResult(orders=["ord-001"])

## The application

Both protected routes get the same registry instance through `connections=`.

In [5]:
def build_app():
    registry = SessionRegistryResource()
    auth = JwtAuthCoordinator(
        secret_key=_SECRET_KEY,
        algorithm=_ALGORITHM,
        role_registry={"viewer": ViewerRole},
        session_id_claim="sid",
    )
    return (
        FastApiAdapter(machine=ActionProductMachine(), auth_coordinator=auth, title="Session Access Rule Demo")
        .post("/auth/login", LoginAction, auth_coordinator=NoAuthCoordinator(context=Context()))
        .post("/auth/logout", LogoutAction, connections={"sessions": registry})
        .get("/orders", ListOrdersAction, connections={"sessions": registry})
        .build()
    )


client = TestClient(build_app(), raise_server_exceptions=False)


def bearer(token: str) -> dict[str, str]:
    return {"Authorization": f"Bearer {token}"}


laptop = client.post("/auth/login", json={"username": "alice"}).json()["access_token"]
phone = client.post("/auth/login", json={"username": "alice"}).json()["access_token"]

## 1. Both sessions are active: the rule allows both

In [6]:
before = client.get("/orders", headers=bearer(laptop))
print(f"GET /orders (laptop, active)  -> {before.status_code} {before.json()}")
assert before.status_code == 200

GET /orders (laptop, active)  -> 200 {'orders': ['ord-001']}


## 2. The laptop session ends; the application records its session id

In [7]:
logout = client.post("/auth/logout", headers=bearer(laptop))
print(f"POST /auth/logout (laptop)    -> {logout.status_code} {logout.json()}")
assert logout.json() == {"ended_session": "session-1"}

POST /auth/logout (laptop)    -> 200 {'ended_session': 'session-1'}


## 3. The laptop token is still valid, but the rule refuses its session

The token is signed and unexpired, so authentication succeeds. The refusal comes from the application's rule, and the adapter answers it with `403`.

In [8]:
after = client.get("/orders", headers=bearer(laptop))
print(f"GET /orders (laptop, ended)   -> {after.status_code} {after.json()}")
assert after.status_code == 403
assert after.json() == {"detail": "Access denied by ACCESS_DECIDE: SESSION_ENDED"}

GET /orders (laptop, ended)   -> 403 {'detail': 'Access denied by ACCESS_DECIDE: SESSION_ENDED'}


## 4. The same user's other session is untouched

In [9]:
other = client.get("/orders", headers=bearer(phone))
print(f"GET /orders (phone, active)   -> {other.status_code} {other.json()}")
assert other.status_code == 200

GET /orders (phone, active)   -> 200 {'orders': ['ord-001']}
